# Etapa 2 - Preparação da Base para o Bandit

Este notebook prepara a base Kaggle `bank-term-deposit-subscription/bank-full.csv`.

- `duration` é removida por leakage temporal.
- A opção `unknown` da coluna `contact` é removida por não representar um braço acionável.
- `cellular` e `telephone` são mantidos como braços.
- `y` é convertida de `yes/no` para `1/0`.
- Os artefatos são salvos em `data/processed/bank-term-deposit-subscription_eda`.

In [36]:
from pathlib import Path
import json

import pandas as pd

In [37]:
RAW_DATA_PATH = Path('../data/raw/bank-term-deposit-subscription/bank-full.csv')
PROCESSED_DIR = Path('../data/processed/bank-term-deposit-subscription_eda')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TREATED_DATA_PATH = PROCESSED_DIR / 'bank_full_tratado.csv'
ARM_STATS_PATH = PROCESSED_DIR / 'arm_stats.csv'
DATA_CONFIG_PATH = PROCESSED_DIR / 'bandit_data_config.json'

In [38]:
if TREATED_DATA_PATH.exists():
    df = pd.read_csv(TREATED_DATA_PATH)
    print(f'Base tratada carregada do cache: {TREATED_DATA_PATH}')
else:
    df = pd.read_csv(RAW_DATA_PATH, sep=';')
    required_columns = {'duration', 'contact', 'y'}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f'Colunas obrigatórias ausentes: {sorted(missing_columns)}')

    leakage_columns = [column for column in ['duration'] if column in df.columns]
    print(f'Colunas de leakage identificadas: {leakage_columns}')
    df = df.drop(columns=leakage_columns).copy()
    df = df[df['contact'].isin(['cellular', 'telephone'])].copy()
    print("Opção 'unknown' removida da coluna arm 'contact'.")

if 'duration' in df.columns:
    raise ValueError('Leakage detectado: duration ainda está presente na base tratada.')
if df['y'].dtype == 'object':
    df['y'] = (df['y'].astype(str).str.lower() == 'yes').astype(int)
else:
    df['y'] = pd.to_numeric(df['y'], errors='coerce')
if df['y'].isna().any() or not set(df['y'].unique()).issubset({0, 1}):
    raise ValueError('O target precisa estar codificado como 0/1.')

df.to_csv(TREATED_DATA_PATH, index=False)

Base tratada carregada do cache: ..\data\processed\bank-term-deposit-subscription_eda\bank_full_tratado.csv


In [39]:
arm_stats = (
    df.groupby('contact', as_index=False)
      .agg(
          observations=('y', 'size'),
          conversions=('y', 'sum'),
          conversion_rate=('y', 'mean')
      )
      .sort_values('conversion_rate', ascending=False)
)
arm_stats.to_csv(ARM_STATS_PATH, index=False)

config = {
    'dataset': 'bank-term-deposit-subscription',
    'source': 'https://www.kaggle.com/datasets/dharmik34/bank-term-deposit-subscription',
    'file': 'bank-full.csv',
    'target': 'y',
    'arm_column': 'contact',
    'arms': arm_stats['contact'].tolist(),
    'leakage_removed': ['duration'],
    'arm_values_removed': ['unknown'],
    'treated_data': str(TREATED_DATA_PATH),
    'arm_stats': str(ARM_STATS_PATH)
}
DATA_CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding='utf-8')

arm_stats

,contact,observations,conversions,conversion_rate
0,cellular,29285,4369,0.149189
1,telephone,2906,390,0.134205


In [40]:
print(f'Dimensões finais: {df.shape[0]:,} linhas x {df.shape[1]} colunas')
print(f'Conversão geral: {df["y"].mean():.2%}')
print(f'Artefatos salvos em: {PROCESSED_DIR}')

Dimensões finais: 32,191 linhas x 16 colunas
Conversão geral: 14.78%
Artefatos salvos em: ..\data\processed\bank-term-deposit-subscription_eda
